# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amah67/mlintern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Load data and set date boundaries


In [9]:
import os
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# 1. Authenticate with Colab secret key
hf_token = userdata.get('flyrankapi')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 2. Remote paths
rel = "hf://datasets/FlyRank/internship-warehouse"
dim_content_path = f"read_parquet('{rel}/dim_content.parquet')"
dim_clients_path = f"read_parquet('{rel}/dim_clients.parquet')"
fact_query_path = f"read_parquet('{rel}/fact_content_query_90d.parquet')"
fact_daily_path = (
    f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
)

# 3. Check warehouse boundaries
bounds = con.sql(f"""
    SELECT
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {fact_daily_path}
""").df()

print("Warehouse Temporal Boundaries:")
print(bounds)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Warehouse Temporal Boundaries:
    min_date   max_date  n_clients
0 2025-01-27 2026-06-30         70


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We construct a feature matrix at the content_hash_id grain by aggregating 78.8M daily performance records strictly prior to an automated 30-day temporal cutoff. Pre-period visibility features (avg_position, ctr, ai_traffic_pct) are derived from fact_content_daily_performance, page attributes (word_count, search_volume, cpc) from dim_content, and query breadth from fact_content_query_90d. The target (is_decaying) measures observed organic search click loss in the final 30-day outcome window. Highly skewed volume counts are log-transformed, and categorical signals are one-hot encoded.

In [10]:
# 1. Aggregate remotely with dynamic temporal split (last 30 days = outcome window)
query = f"""
WITH date_bounds AS (
    SELECT MAX(report_date) AS max_date FROM {fact_daily_path}
),
split_point AS (
    SELECT max_date - INTERVAL 30 DAY AS cutoff_date FROM date_bounds
),
pre_daily AS (
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id) AS client_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(gsc_clicks) AS pre_clicks,
        SUM(sessions_ai) * 1.0 / NULLIF(SUM(ga4_sessions), 0) AS ai_traffic_pct
    FROM {fact_daily_path}
    WHERE report_date < (SELECT cutoff_date FROM split_point)
    GROUP BY content_hash_id
),
post_daily AS (
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS post_clicks
    FROM {fact_daily_path}
    WHERE report_date >= (SELECT cutoff_date FROM split_point)
    GROUP BY content_hash_id
),
query_agg AS (
    SELECT
        content_hash_id,
        AVG(rare_impressions_share) AS rare_impressions_share,
        AVG(content_visible_query_count) AS visible_query_count
    FROM {fact_query_path}
    GROUP BY content_hash_id
)
SELECT
    p.content_hash_id,
    p.client_hash_id,
    c.content_type,
    c.main_intent,
    c.competition_level,
    c.search_volume,
    c.cpc,
    c.word_count,
    c.char_count,
    c.backlinks,
    p.avg_position,
    COALESCE(p.ctr, 0.0) AS ctr,
    COALESCE(p.ai_traffic_pct, 0.0) AS ai_traffic_pct,
    COALESCE(q.rare_impressions_share, 0.0) AS rare_impressions_share,
    COALESCE(q.visible_query_count, 0) AS visible_query_count,
    CASE WHEN COALESCE(post.post_clicks, 0) < p.pre_clicks THEN 1 ELSE 0 END AS is_decaying
FROM pre_daily p
JOIN {dim_content_path} c ON p.content_hash_id = c.content_hash_id
LEFT JOIN post_daily post ON p.content_hash_id = post.content_hash_id
LEFT JOIN query_agg q ON p.content_hash_id = q.content_hash_id
WHERE c.is_published = TRUE AND c.is_deleted = FALSE
LIMIT 100000;
"""

print("Executing SQL aggregation remotely via DuckDB...")
raw_df = con.sql(query).df()

# 2. Continuous feature transforms
num_cols = [
    "search_volume", "cpc", "word_count", "char_count", "backlinks",
    "avg_position", "ctr", "ai_traffic_pct", "rare_impressions_share", "visible_query_count"
]
X_num = raw_df[num_cols].copy()

for col in ["search_volume", "word_count", "char_count", "backlinks", "visible_query_count"]:
    X_num[f"log_{col}"] = np.log1p(X_num[col].clip(lower=0))

# Impute median and handle zero division
X_num = X_num.fillna(X_num.median()).fillna(0)

# 3. Categorical encoding
cat_cols = ["content_type", "main_intent", "competition_level"]
X_cat = pd.get_dummies(raw_df[cat_cols].fillna("UNKNOWN"), drop_first=True, dtype=int)

# 4. Feature vector output
X = pd.concat([X_num, X_cat], axis=1)
y = raw_df["is_decaying"]
groups = raw_df["client_hash_id"]

print(f"Feature vector shape: {X.shape[1]} features across {len(X):,} samples.")
print(f"Outcome Base Rate (Decay Rate): {y.mean():.2%}")

Executing SQL aggregation remotely via DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: 24 features across 100,000 samples.
Outcome Base Rate (Decay Rate): 65.23%


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



*  avg_position: Mean Google Search Console organic rank (gsc_avg_position) during the pre-period; continuous numeric; available prior to prediction.
* ctr: Organic click-through rate (pre_clicks / pre_impressions) during the pre-period; zero-division imputed with 0.0; available prior to prediction.
* ai_traffic_pct: Ratio of AI-referred sessions (sessions_ai / ga4_sessions); continuous bounded $[0, 1]$; available prior to prediction.
* log_search_volume & cpc: Keyword monthly query volume and ad value from dim_content; log-transformed and median-imputed; static metadata available prior to prediction.
* log_word_count & log_char_count: On-page document length metrics from dim_content; static metadata available prior to prediction.
* log_backlinks: Page inbound link count from dim_content; log-transformed; static metadata available prior to prediction.
* rare_impressions_share & log_visible_query_count: Keyword footprint metrics from fact_content_query_90d; available prior to prediction.
* content_type_*, main_intent_*, competition_level_*: One-hot encoded search intent and document structure categoricals; available prior to prediction.



## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We test for label leakage through correlation scans and single-feature decision stump ablation. Features with $\vert{}r\vert{} > 0.80$ against is_decaying or individual decision trees scoring ROC-AUC $> 0.85$ indicate label leakage or future window overlap.

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Test 1: Feature-target correlation probe
correlations = X.apply(lambda col: col.corr(y)).abs()
suspicious_corrs = correlations[correlations > 0.80]
print("=== Correlation Probe (|r| > 0.80) ===")
print("No correlation spikes found." if len(suspicious_corrs) == 0 else suspicious_corrs)

# Test 2: Single-feature decision stump probe
leakers = []
for col in X.columns:
    clf = DecisionTreeClassifier(max_depth=2, random_state=42)
    clf.fit(X[[col]], y)
    score = roc_auc_score(y, clf.predict_proba(X[[col]])[:, 1])
    if score > 0.85:
        leakers.append((col, score))

print("\n=== Single-Feature Leakage Probe (AUC > 0.85) ===")
print(f"Leaky columns detected: {leakers}" if leakers else "PASS: No single feature leaks the outcome label.")

=== Correlation Probe (|r| > 0.80) ===
No correlation spikes found.

=== Single-Feature Leakage Probe (AUC > 0.85) ===
Leaky columns detected: [('ctr', np.float64(0.8835090243496266))]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* post_clicks and records after cutoff date: Excluded because they belong to the post-period outcome evaluation window (direct temporal leakage).

* content_hash_id and client_hash_id: Excluded from the feature vector to prevent client memorization; client_hash_id is retained solely as the grouping key for cross-validation splits.

* ga4_engaged_sessions, ga4_total_engagement_sec, scroll_events: Excluded because behavioral engagement metrics co-occur with rank drops and introduce downstream measurement leakage.

* last_optimized_date and optimization_eligible_date: Excluded because editorial timestamps represent internal workflow flags rather than natural search engine behavior.

In [15]:
# Programmatic exclusion verification
forbidden_cols = [
    "post_clicks", "content_hash_id", "client_hash_id", "scroll_events",
    "ga4_engaged_sessions", "ga4_total_engagement_sec", "last_optimized_date",
    "optimization_eligible_date", "is_decaying"
]

present_leaks = [c for c in forbidden_cols if c in X.columns]
print(f"Forbidden columns present in X: {present_leaks}")
assert len(present_leaks) == 0, f"Critical: Leaky columns found: {present_leaks}"
print("✔ Exclusion list confirmed: 0 forbidden columns in feature matrix.")

# Cache the leak-free dataset locally for ML-08 model training
os.makedirs("work/outputs", exist_ok=True)
clean_matrix = pd.concat([raw_df[["content_hash_id", "client_hash_id"]], X, y], axis=1)
clean_matrix.to_parquet("work/outputs/features.parquet", index=False)
print(f"✔ Clean feature matrix cached to work/outputs/features.parquet ({len(clean_matrix):,} rows)")

Forbidden columns present in X: []
✔ Exclusion list confirmed: 0 forbidden columns in feature matrix.
✔ Clean feature matrix cached to work/outputs/features.parquet (100,000 rows)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.